In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
import time

base = "/Volumes/workspace/default/m5/"

# Fact table from notebook 01 — already long, partitioned by store_id
long = spark.read.parquet(base + "out/sales_long")

# Dimension tables with explicit schemas (lesson from 01)
cal_schema = StructType([
    StructField("date", StringType()), StructField("wm_yr_wk", IntegerType()),
    StructField("weekday", StringType()), StructField("wday", IntegerType()),
    StructField("month", IntegerType()), StructField("year", IntegerType()),
    StructField("d", StringType()),
    StructField("event_name_1", StringType()), StructField("event_type_1", StringType()),
    StructField("event_name_2", StringType()), StructField("event_type_2", StringType()),
    StructField("snap_CA", IntegerType()), StructField("snap_TX", IntegerType()), StructField("snap_WI", IntegerType()),
])
prices_schema = StructType([
    StructField("store_id", StringType()), StructField("item_id", StringType()),
    StructField("wm_yr_wk", IntegerType()), StructField("sell_price", FloatType()),
])

cal    = spark.read.csv(base + "calendar.csv",    header=True, schema=cal_schema) \
              .select("d", "date", "wm_yr_wk", "weekday", "event_name_1", "snap_CA", "snap_TX", "snap_WI")
prices = spark.read.csv(base + "sell_prices.csv", header=True, schema=prices_schema)

print(f"long   {long.count():>12,}")
print(f"cal    {cal.count():>12,}")
print(f"prices {prices.count():>12,}")

long     59,181,090
cal           1,969
prices    6,841,121


In [0]:
# ---- Baseline: two sort-merge joins (forced via hint) ----------------------
# Join 1: long ⋈ cal on d          (59M × 1,969)   → adds wm_yr_wk, date
# Join 2: result ⋈ prices on (store_id, item_id, wm_yr_wk)   (59M × 6.8M)
#
# Without the hint, Spark's AQE would auto-broadcast `cal` (< 10 MB) and we'd
# never see the shuffle cost. `merge` forces SortMergeJoin so we have a real baseline.

def timed(df, label):
    t0 = time.time(); n = df.count(); dt = time.time() - t0
    print(f"{label:<32} {n:>12,} rows   {dt:6.1f}s")
    return dt

joined_smj = (long
    .join(cal.hint("merge"), "d")
    .join(prices.hint("merge"), ["store_id", "item_id", "wm_yr_wk"]))

t_smj = timed(joined_smj, "A  sort-merge × 2")
joined_smj.explain()     # look for two `SortMergeJoin` + `Exchange hashpartitioning` blocks

A  sort-merge × 2                  46,881,677 rows     32.7s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   Project [store_id#48378, item_id#48372, wm_yr_wk#48385, d#48376, id#48371, dept_id#48373, cat_id#48374, state_id#48375, sales#48377, date#48384, weekday#48386, event_name_1#48391, snap_CA#48395, snap_TX#48396, snap_WI#48397, sell_price#48408]
   +- SortMergeJoin [store_id#48378, item_id#48372, wm_yr_wk#48385], [store_id#48405, item_id#48406, wm_yr_wk#48407], Inner
      :- Sort [store_id#48378 ASC NULLS FIRST, item_id#48372 ASC NULLS FIRST, wm_yr_wk#48385 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(store_id#48378, item_id#48372, wm_yr_wk#48385, 237), ENSURE_REQUIREMENTS, [plan_id=12817]
      :     +- Project [d#48376, id#48371, item_id#48372, dept_id#48373, cat_id#48374, state_id#48375, sales#48377, store_id#48378, date#48384, wm_yr_wk#48385, weekday#48386, event_name_1#48391, snap_CA#48395, snap_TX#48396, snap_WI#48397]
   

In [0]:
# ---- Variant B: broadcast the small dimension --------------------------------
# `cal` is 1,969 rows (~100 KB). Broadcasting ships one copy to every executor,
# so join 1 becomes a map-side BroadcastHashJoin with no shuffle of the 59M-row side.
# `prices` (6.8M rows, ~200 MB) is too large to broadcast sensibly — join 2 stays sort-merge.

joined_bc = (long
    .join(F.broadcast(cal), "d")
    .join(prices.hint("merge"), ["store_id", "item_id", "wm_yr_wk"]))

t_bc = timed(joined_bc, "B  broadcast cal + SMJ prices")
joined_bc.explain()      # join 1 should now read `BroadcastHashJoin`; one fewer Exchange

B  broadcast cal + SMJ prices      46,881,677 rows     10.0s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   Project [store_id#48378, item_id#48372, wm_yr_wk#48385, d#48376, id#48371, dept_id#48373, cat_id#48374, state_id#48375, sales#48377, date#48384, weekday#48386, event_name_1#48391, snap_CA#48395, snap_TX#48396, snap_WI#48397, sell_price#48408]
   +- SortMergeJoin [store_id#48378, item_id#48372, wm_yr_wk#48385], [store_id#48405, item_id#48406, wm_yr_wk#48407], Inner
      :- ColumnarToRow
      :  +- PhotonResultStage
      :     +- PhotonSort [store_id#48378 ASC NULLS FIRST, item_id#48372 ASC NULLS FIRST, wm_yr_wk#48385 ASC NULLS FIRST]
      :        +- PhotonShuffleExchangeSource false
      :           +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#13704]
      :              +- PhotonShuffleExchangeSink hashpartitioning(store_id#48378, item_id#48372, wm_yr_wk#48385, 237)
      :                 +- PhotonProject [d#48376, id#48371, item_id#

In [0]:
# ---- Variant C: no hints, let AQE choose -------------------------------------
# Serverless enables Adaptive Query Execution. With no hints, AQE should
# broadcast `cal` automatically and pick a strategy for `prices` based on
# runtime stats. This is the "what production would do" reference point.

joined_aqe = (long
    .join(cal, "d")
    .join(prices, ["store_id", "item_id", "wm_yr_wk"]))

t_aqe = timed(joined_aqe, "C  no hints (AQE decides)")
joined_aqe.explain()

# ---- Data-quality observation: inner join drops rows -------------------------
# prices only lists weeks an item was actually sold. Inner-joining drops
# pre-launch days. Left join keeps them with sell_price = NULL.
n_long  = long.count()
n_inner = joined_aqe.count()
print(f"\nrows before join {n_long:,}  after inner join {n_inner:,}  dropped {n_long-n_inner:,} ({(n_long-n_inner)/n_long:.1%})")

print(f"\nSummary   A={t_smj:.1f}s   B={t_bc:.1f}s   C={t_aqe:.1f}s")

C  no hints (AQE decides)          46,881,677 rows      4.8s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [store_id#48378, item_id#48372, wm_yr_wk#48385, d#48376, id#48371, dept_id#48373, cat_id#48374, state_id#48375, sales#48377, date#48384, weekday#48386, event_name_1#48391, snap_CA#48395, snap_TX#48396, snap_WI#48397, sell_price#48408]
         +- PhotonShuffledHashJoin [store_id#48378, item_id#48372, wm_yr_wk#48385], [store_id#48405, item_id#48406, wm_yr_wk#48407], Inner, BuildRight
            :- PhotonShuffleExchangeSource false
            :  +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#14493]
            :     +- PhotonShuffleExchangeSink hashpartitioning(store_id#48378, item_id#48372, wm_yr_wk#48385, 237)
            :        +- PhotonProject [d#48376, id#48371, item_id#48372, dept_id#48373, cat_id#48374, state_id#48375, sales#48377, store_id#48378, date#48384, wm_yr_wk

In [0]:
# ---- Canonical joined table: LEFT join keeps pre-launch days ----------------
# prices.csv only lists weeks an item was on sale, so an inner join silently
# drops pre-launch days (12.3M rows, 20.8%). Keep them; sell_price = NULL and
# is_listed = False mean "not yet on shelf". Filtering is the analyst's call.

joined = (long
    .join(cal, "d", "left")
    .join(prices, ["store_id", "item_id", "wm_yr_wk"], "left")
    .withColumn("is_listed", F.col("sell_price").isNotNull()))

# Sanity check: if the dropped rows really are pre-launch, sales there must be 0.
joined.groupBy("is_listed").agg(
    F.count("*").alias("rows"),
    F.sum("sales").alias("total_sales"),
).orderBy("is_listed").show()
# expect: False ≈ 12.3M rows, total_sales = 0 ; True ≈ 46.9M rows

# Persist
out_joined = base + "out/sales_joined"
t0 = time.time()
joined.write.mode("overwrite").partitionBy("store_id").parquet(out_joined)
print(f"written {joined.count():,} rows — {time.time()-t0:.1f}s")

+---------+--------+-----------+
|is_listed|    rows|total_sales|
+---------+--------+-----------+
|    false|12299413|          0|
|     true|46881677|   66927173|
+---------+--------+-----------+

written 59,181,090 rows — 32.5s


In [0]:
joined.filter(~F.col("is_listed")).agg(
    F.count("*").alias("unlisted_rows"),
    F.sum("sales").alias("sales_while_unlisted")   # 預期 0
).show()

+-------------+--------------------+
|unlisted_rows|sales_while_unlisted|
+-------------+--------------------+
|     12299413|                   0|
+-------------+--------------------+

